# Day 3, Notebook 1: profile before you touch

Yesterday somebody told you which orders were dirty. Today nobody tells you, and the first
question is how many usable orders this file actually has.

Run this notebook top to bottom. The failure in it is deliberate and it does not raise, which
is the point: today's mistakes print a number and look fine.

Position in the day:

`[profile the columns] > [decide per field] > [find the hidden rows] > [investigate the extremes] > [ship with the log]`

The map below is where this notebook sits in the day and what it adds. Every
notebook in the programme opens on the same pair, so you know where you are before you read a line.

The cell that draws it also brings in the programme's helper. `scripts/c2kit.py` is found by
walking up from this notebook's own folder, which is what lets the same file run whether you
pressed Run All here or a script ran it for you. The helper loads the day's data from `../data/`,
draws every diagram you see in these notebooks, and runs the checks that tell you a cell did what
it claimed.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["profiling the columns", "the rows a profile cannot see", "hands-on: the full pass", "hands-on: which dataset"], lit=0, title="the day's notebooks", show=False),
    kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], title="what this notebook adds", show=False),
)

## Setup

`csv` is today's tool. `traceback` is plumbing, as yesterday.

In [2]:
import csv
import traceback

DATA_DIR = "../data"
ORDERS_CSV = f"{DATA_DIR}/C2_W01_D03_orders_STUDENT.csv"

with open(ORDERS_CSV) as f:
    orders = list(csv.DictReader(f))

FIELDS = list(orders[0].keys())

print(f"{len(orders)} orders loaded")
print("fields:", FIELDS)
print(orders[0])

50 orders loaded
fields: ['order_id', 'customer_id', 'segment', 'amount', 'status', 'order_date', 'discount']
{'order_id': 'KR4200', 'customer_id': 'C1451', 'segment': 'Student', 'amount': '1280', 'status': 'cancelled', 'order_date': '2026-08-03', 'discount': '150'}


In [3]:
kit.check("fifty rows came off the file", len(orders) == 50, f"got {len(orders)}")
kit.check("seven fields per row", len(orders[0]) == 7, ", ".join(orders[0]))
kit.check("the amount column is where today's work is",
          sum(1 for r in orders if not r["amount"].strip().lstrip("-").isdigit()) == 6,
          "six values will not convert")

## The functions you wrote yesterday

Copied across without one character changed. Read them and confirm that nothing in them knows
whether it is handling one order or fifty.

In [4]:
def normalise_amount(raw):
    """Convert an amount, or raise ValueError with the interpreter's own wording."""
    return int(raw)


def clean_record(record):
    """Return one order with its amount as a number, or raise ValueError saying what arrived."""
    keeper = dict(record)
    keeper["amount"] = normalise_amount(record["amount"])
    return keeper


def clean_records(rows):
    """Call clean_record on every row and keep the failures, each with its reason."""
    clean = []
    rejects = []
    for r in rows:
        try:
            clean.append(clean_record(r))
        except ValueError as e:
            rejects.append({"order_id": r["order_id"], "reason": str(e)})
    return clean, rejects

print("carried forward:", normalise_amount.__name__, clean_record.__name__, clean_records.__name__)

carried forward: normalise_amount clean_record clean_records


In [5]:
kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], lit=0)

## Section 1: Monday's counter

Monday you counted how many orders had a value in a field. Here it is again, unchanged in spirit.

In [6]:
def count_present(rows, field):
    """How many rows have anything at all in this field."""
    return sum(1 for r in rows if r[field].strip() != "")

for field in FIELDS:
    print(f"{field:12} present {count_present(orders, field)}/{len(orders)}")

order_id     present 50/50
customer_id  present 50/50
segment      present 50/50
amount       present 48/50
status       present 50/50
order_date   present 50/50
discount     present 11/50


In [7]:
kit.check("amount is present on 48 of the 50 rows", count_present(orders, "amount") == 48,
          f'{count_present(orders, "amount")} of 50')
kit.check("discount is present on 11", count_present(orders, "discount") == 11)
kit.check("every other field is present on all fifty",
          all(count_present(orders, f) == 50 for f in orders[0]
              if f not in ("amount", "discount")))

Two fields are already interesting. `amount` is not filled in everywhere, and `discount` is
mostly empty.

That is as far as presence takes you. A field can be present and still useless.

In [8]:
kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], lit=1)

## Section 2: presence plus convertibility

Section 1 plus one new element: can the value become the type you need.

```
present     the box has something in it
converts    what is in the box is usable
```

The gap between those two numbers is your work list.

In [9]:
def count_converts(rows, field):
    """How many rows hold a value in this field that becomes an integer."""
    n = 0
    for r in rows:
        try:
            normalise_amount(r[field])
        except ValueError:
            continue
        n += 1
    return n

present = count_present(orders, "amount")
converts = count_converts(orders, "amount")
print(f"amount   present {present}/{len(orders)}   converts {converts}/{len(orders)}")
print(f"values that are there and unusable: {present - converts}")

amount   present 48/50   converts 44/50
values that are there and unusable: 4


In [10]:
kit.check("44 amounts convert", count_converts(orders, "amount") == 44,
          f'{count_converts(orders, "amount")} of 50')
kit.check("so four values are present and unusable",
          count_present(orders, "amount") - count_converts(orders, "amount") == 4,
          "the two empties are absent rather than unusable")

### The three counts, drawn

In [11]:
kit.flow(["present: the box has something",
          "converts: what is in it is usable",
          "distinct: what kind of field this is"], lit=1,
         title="one column, three questions")

Four orders have an amount that looks filled in and will not convert. Those are the ones that
would have slipped past a presence check, which is exactly why presence alone is not a profile.

In [12]:
for r in orders:
    try:
        normalise_amount(r["amount"])
    except ValueError as e:
        print(f'{r["order_id"]}  {r["amount"]!r:16} {e}')

KR4210  'twelve'         invalid literal for int() with base 10: 'twelve'
KR4214  ''               invalid literal for int() with base 10: ''
KR4231  '12,400'         invalid literal for int() with base 10: '12,400'
KR4235  '24 500'         invalid literal for int() with base 10: '24 500'
KR4237  ''               invalid literal for int() with base 10: ''
KR4240  'Rs 8000'        invalid literal for int() with base 10: 'Rs 8000'


A word, two empty cells, a thousands separator, an internal space and a currency prefix. Six
different ways for the same column to be unreadable, and every one of them arrived from a system
somebody else runs.

In [13]:
kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], lit=2)

## Section 3: the third count

Section 2 plus one new element: how many different values a field holds.

`distinct` tells you what kind of field you are looking at, and it is the only one of the three
that can fall when somebody damages the data.

In [14]:
def profile_field(rows, field):
    """The three counts, for one field."""
    return {
        "present": count_present(rows, field),
        "converts": count_converts(rows, field),
        "distinct": len({r[field] for r in rows}),
    }


def print_profile(rows, label):
    print(f"{label}   ({len(rows)} rows)")
    print(f"  {'field':12} {'present':>9} {'converts':>9} {'distinct':>9}")
    for field in FIELDS:
        p = profile_field(rows, field)
        print(f"  {field:12} {p['present']:>9} {p['converts']:>9} {p['distinct']:>9}")

print_profile(orders, "RAW")

RAW   (50 rows)
  field          present  converts  distinct
  order_id            50         0        49
  customer_id         50         0        47
  segment             50         0         4
  amount              48        44        46
  status              50         0         3
  order_date          50         0        20
  discount            11        11         9


In [15]:
kit.table(
    ["field", "present", "converts", "distinct", "what the shape says"],
    [[f, profile_field(orders, f)["present"], profile_field(orders, f)["converts"],
      profile_field(orders, f)["distinct"],
      "an id, and one value repeats" if f == "order_id" else
      "a category" if profile_field(orders, f)["distinct"] <= 5 else
      "a measure with work to do" if f == "amount" else
      "absent by design" if f == "discount" else "a free value"]
     for f in orders[0]],
    caption="The whole file, profiled, before anything is touched",
)
kit.check("order_id holds 49 distinct values across 50 rows",
          profile_field(orders, "order_id")["distinct"] == 49)
kit.check("segment is a category with four values",
          profile_field(orders, "segment")["distinct"] == 4)

field,present,converts,distinct,what the shape says
order_id,50,0,49,"an id, and one value repeats"
customer_id,50,0,47,a free value
segment,50,0,4,a category
amount,48,44,46,a measure with work to do
status,50,0,3,a category
order_date,50,0,20,a free value
discount,11,11,9,absent by design


Read it as a shape rather than as numbers.

`order_id` has 49 distinct values across 50 rows. Hold that thought; notebook 2 is about it.

`segment` has 4 distinct values, so it is a category. `status` has 3. `customer_id` has 47, so it
is close to unique. `order_date` has 20, so orders cluster on days.

`amount` has 46 distinct values and converts on only 44 of 50.

### Milestone: where this shows up in production

Every ingestion service you will work on runs a profile like this on arrival and refuses the batch
when a count moves outside its expected range. The alert that fires at three in the morning is
usually a `present` count that dropped, which means an upstream system quietly stopped sending a
field.

### Interview question this milestone just made answerable

"What do you look at first when a dataset arrives?"

Three counts per field, and the gap between present and converts, before changing anything.

In [16]:
kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], lit=3)

## Section 4: the deliberate failure

Section 3 plus one new element: what happens when somebody helps.

The cell below is a reasonable-looking cleaning pass. It replaces every failure with a stated
default of zero, so nothing is ever rejected. Read it, then look at what it does to the profile.

In [17]:
def coerce_everything(rows):
    """Replace anything that will not convert with a stated default of zero."""
    out = []
    for r in rows:
        c = dict(r)
        try:
            c["amount"] = str(normalise_amount(r["amount"]))
        except ValueError:
            c["amount"] = "0"
        c["discount"] = r["discount"] if r["discount"].strip() else "0"
        out.append(c)
    return out

coerced = coerce_everything(orders)
print_profile(coerced, "COERCED")

COERCED   (50 rows)
  field          present  converts  distinct
  order_id            50         0        49
  customer_id         50         0        47
  segment             50         0         4
  amount              50        50        42
  status              50         0         3
  order_date          50         0        20
  discount            50        50         9


Nothing raised. Nothing was rejected. Compare the two profiles side by side.

In [18]:
print(f"  {'field':12} {'RAW present':>12} {'converts':>9} {'distinct':>9}  |  "
      f"{'COERCED present':>16} {'converts':>9} {'distinct':>9}")
for field in FIELDS:
    a = profile_field(orders, field)
    b = profile_field(coerced, field)
    moved = "   <-- moved" if a != b else ""
    print(f"  {field:12} {a['present']:>12} {a['converts']:>9} {a['distinct']:>9}  |  "
          f"{b['present']:>16} {b['converts']:>9} {b['distinct']:>9}{moved}")

  field         RAW present  converts  distinct  |   COERCED present  converts  distinct
  order_id               50         0        49  |                50         0        49
  customer_id            50         0        47  |                50         0        47
  segment                50         0         4  |                50         0         4
  amount                 48        44        46  |                50        50        42   <-- moved
  status                 50         0         3  |                50         0         3
  order_date             50         0        20  |                50         0        20
  discount               11        11         9  |                50        50         9   <-- moved


Every count moved the way you want counts to move, and the dataset got worse.

`discount` went from 11 present to 50. Thirty nine orders now carry a discount that nobody ever
recorded, and no column on this page says which thirty nine.

`amount` went from 44 converting to 50. Six unreadable values became the number zero.

### The count that tells you

`amount` distinct fell from 46 to 42. That is the only number that went down, and it went down
because five different broken values collapsed into a single zero.

A count that can only rise cannot warn you. Watch the one that can fall.

In [19]:
lost = {r["amount"] for r in orders} - {r["amount"] for r in coerced}
print("raw values that no longer exist anywhere in the coerced data:")
for v in sorted(lost):
    print("  ", repr(v))
print()
print("orders whose amount is now zero:",
      sum(1 for r in coerced if r["amount"] == "0"))
print("orders whose amount was genuinely zero before:",
      sum(1 for r in orders if r["amount"] == "0"))

raw values that no longer exist anywhere in the coerced data:
   ''
   '12,400'
   '24 500'
   'Rs 8000'
   'twelve'

orders whose amount is now zero: 6
orders whose amount was genuinely zero before: 0


In [20]:
kit.check("the coerced pass raised nothing and rejected nothing", len(coerced) == len(orders))
kit.check("six rows were replaced by a zero nobody wrote, across five distinct raw values",
          len(lost) == 5, "the two empty cells share one raw value: " + ", ".join(sorted(lost)))
kit.check("distinct fell, which is the only count that can warn you",
          len({r["amount"] for r in coerced}) < len({r["amount"] for r in orders}))

### What the coerce-everything pass did to each count

In [21]:
kit.matrix(["present", "converts", "distinct"],
           ["raw", "coerced", "what that means"],
           [["48", "50", "rose, and looks like progress"],
            ["44", "50", "rose, and looks like progress"],
            ["46", "41", "fell, and is the only warning you get"]],
           title="every count moved the way you want, and the file got worse")

Six orders now read zero and none of them ever did. If somebody computes an average tomorrow, those
six pull it down and there is nothing left in the file to say so.

### Milestone: where this shows up in production

HGNC, 2020. About 27 human genes were formally renamed because spreadsheets silently coerced names
like SEPT1 into dates, after a 2016 audit found gene-name errors in roughly a fifth of genetics
papers with spreadsheet supplements.

Nobody chose to corrupt anything. A default was applied quietly, at scale, for years.

### Interview question this milestone just made answerable

"You coerced every failure to a default and the dataset looks clean. What did you lose?"

The evidence. The count of what failed, which values failed, and the ability to tell a real zero
from a rescued one.

In [22]:
kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], lit=4)

## Section 5: the honest pass

Section 4 plus one new element: the same work, with the failures kept.

`clean_records` is yesterday's function, unedited. Point it at fifty orders and it does the right
thing, because you wrote it to reject rather than to guess.

In [23]:
clean, rejects = clean_records(orders)
print(f"input {len(orders)}, clean {len(clean)}, rejected {len(rejects)}")
for r in rejects:
    print("  ", r)
print()
assert len(clean) + len(rejects) == len(orders), "orders went missing"
print(f"{len(orders)} in = {len(clean)} clean + {len(rejects)} rejected")

input 50, clean 44, rejected 6
   {'order_id': 'KR4210', 'reason': "invalid literal for int() with base 10: 'twelve'"}
   {'order_id': 'KR4214', 'reason': "invalid literal for int() with base 10: ''"}
   {'order_id': 'KR4231', 'reason': "invalid literal for int() with base 10: '12,400'"}
   {'order_id': 'KR4235', 'reason': "invalid literal for int() with base 10: '24 500'"}
   {'order_id': 'KR4237', 'reason': "invalid literal for int() with base 10: ''"}
   {'order_id': 'KR4240', 'reason': "invalid literal for int() with base 10: 'Rs 8000'"}

50 in = 44 clean + 6 rejected


In [24]:
kit.check("input equals clean plus rejected",
          len(clean) + len(rejects) == len(orders),
          f"{len(orders)} in, {len(clean)} clean, {len(rejects)} rejected")
kit.check("the profile predicted the rejection count", len(rejects) == 6)
kit.check("every rejection carries the interpreter's own wording",
          all(r.get("reason") for r in rejects))

Same six failures the profile predicted, each carrying the interpreter's own wording, and the
reconciliation holds.

Compare that with the coerced pass, which reported nothing at all.

In [25]:
kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], lit=5)

## Section 6: missing is a decision

Two fields are incomplete in this file and they get opposite treatment. Both decisions are written
down, which is what makes them decisions rather than habits.

In [26]:
decisions = []

def record_decision(field, finding, choice, reason):
    """Every cleaning act becomes one line a reviewer can follow."""
    decisions.append({"field": field, "finding": finding, "choice": choice, "reason": reason})

record_decision("amount",
                f"{count_present(orders,'amount')}/{len(orders)} present, "
                f"{count_converts(orders,'amount')}/{len(orders)} convertible",
                "reject the unusable orders",
                "amount is required, and an order with no readable amount cannot be computed on")

record_decision("discount",
                f"{count_present(orders,'discount')}/{len(orders)} present",
                "keep absent as absent",
                "absence means no discount was applied, which is a fact rather than a gap")

for d in decisions:
    print(f"{d['field']:9} {d['finding']:42} -> {d['choice']}")
    print(f"{'':9} because {d['reason']}")

amount    48/50 present, 44/50 convertible           -> reject the unusable orders
          because amount is required, and an order with no readable amount cannot be computed on
discount  11/50 present                              -> keep absent as absent
          because absence means no discount was applied, which is a fact rather than a gap


The second one is worth arguing about. Filling `discount` with zero is defensible, and it costs you
the ability to ever tell an order with no discount from an order whose discount was zero.

You are allowed to make that trade. You are not allowed to make it silently.

### Interview question

"Drop, default, or keep and flag. Match each to its one-line reason."

Drop when the field is required and the record is unusable without it. Default when absence
genuinely means something and you can say what. Keep and flag when you need the record and the gap
has to travel with it.

### What this notebook established

In [27]:
kit.table(
    ["The idea", "What proved it here"],
    [["Profile before you have permission to change anything", "three counts on all seven fields, before any cleaning"],
     ["present minus converts is the dangerous group", "48 present, 44 converts, four values that look fine and are not"],
     ["A coerce-everything pass destroys the evidence", "six raw values gone, and only distinct fell to say so"],
     ["Every cleaning act is a line in the decisions log", "two decisions recorded with their reasons"]],
    caption="Day 3, notebook 1",
)
kit.flow(["Monday's counter", "presence plus convertibility", "the third count", "the coerce-everything trap", "the honest pass", "missing is a decision"], lit=5, title="the notebook, end to end")
kit.check_summary()

The idea,What proved it here
Profile before you have permission to change anything,"three counts on all seven fields, before any cleaning"
present minus converts is the dangerous group,"48 present, 44 converts, four values that look fine and are not"
A coerce-everything pass destroys the evidence,"six raw values gone, and only distinct fell to say so"
Every cleaning act is a line in the decisions log,two decisions recorded with their reasons


## Crux

Profiling is what you do before you have permission to change anything.

Three counts per field, and the gap between present and converts is your work list.

A count that only rises cannot warn you.

## What notebook 2 does with this

`order_id` has 49 distinct values across 50 rows, and nothing so far has explained that.

Notebook 2 is about the one row this profile cannot see.